# 03. Preprocessing

> Standard ECoG cleaning pipeline: notch filter for line-noise harmonics, bandpass to the relevant frequencies, and common average referencing across the 60-channel grid.

Each step is exposed as a reusable function and validated by plotting before/after spectra and time-domain traces. Bad-channel rejection (informed by `02_eda`) is applied here before re-referencing.

In [ ]:
#| default_exp preprocessing

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np
from scipy.signal import iirnotch, butter, filtfilt

## Notch filter

Remove line-noise harmonics (50 Hz in Europe, 60 Hz in the US) with a zero-phase IIR notch. `harmonics=N` cascades the notch at `f, 2f, ... Nf` so a single call covers the fundamental and its overtones.

In [ ]:
#| export
def notch_filter(x, fs, freq=50.0, Q=30.0, harmonics=3):
    """Zero-phase IIR notch at `freq` (and `harmonics` overtones) on a (channels, time) array."""
    out = x
    for k in range(1, harmonics + 1):
        f = freq * k
        if f >= fs / 2: break
        b, a = iirnotch(f, Q, fs=fs)
        out = filtfilt(b, a, out, axis=-1)
    return out

## Bandpass

Zero-phase Butterworth bandpass. The default `1–200 Hz` range removes the DC drift introduced by the DC-coupled recording (low end) while preserving the high-gamma band (70–170 Hz) that carries most of the motor-decoding signal in ECoG.

In [ ]:
#| export
def bandpass(x, fs, low=1.0, high=200.0, order=4):
    """Zero-phase Butterworth bandpass on a (channels, time) array."""
    b, a = butter(order, [low, high], btype='band', fs=fs)
    return filtfilt(b, a, x, axis=-1)

## Common Average Reference (CAR)

Subtract the across-channel mean from each sample. This suppresses noise that is common to the whole grid (movement artifacts, distant muscle activity) and is the canonical re-reference for grid ECoG.

In [ ]:
#| export
def common_average_reference(x):
    """Subtract the across-channel mean from each sample of `x` (channels, time)."""
    return x - x.mean(axis=0, keepdims=True)

## Bad-channel rejection

Mark channels whose std is more than `threshold` median-absolute-deviations away from the median std. Useful before CAR so the mean isn't dragged around by a single saturated electrode.

In [ ]:
#| export
def find_bad_channels(x, threshold=5.0):
    """Return the indices of channels whose std is `threshold` MADs from the median std."""
    s = x.std(axis=1)
    med = np.median(s)
    mad = np.median(np.abs(s - med)) + 1e-12
    return np.where(np.abs(s - med) / mad > threshold)[0]

## Convenience pipeline

`preprocess(ecog, fs)` chains everything in the recommended order: notch (with harmonics) → bandpass → CAR. Bad channels are zeroed out before CAR so they don't bias the across-channel mean.

In [ ]:
#| export
def preprocess(ecog, fs, line_freq=50.0, line_harmonics=3, band=(1.0, 200.0), bad_threshold=5.0):
    """Notch (with harmonics) → bandpass → zero out bad channels → CAR."""
    y = notch_filter(ecog, fs, freq=line_freq, harmonics=line_harmonics)
    y = bandpass(y, fs, *band)
    bad = find_bad_channels(y, threshold=bad_threshold)
    y[bad] = 0.0
    y = common_average_reference(y)
    return y, bad

## Visual validation

PSD and a single-channel time-domain trace, before vs after preprocessing. Run interactively to confirm the notch removed line-noise peaks, bandpass killed the DC drift, and CAR didn't introduce ringing.

In [ ]:
%config InlineBackend.figure_format = 'retina'

import matplotlib.pyplot as plt
from scipy.signal import welch
from br41n_ecog_hand_pose.data import load_ecog

plt.rcParams.update({
    'axes.grid':      True,
    'grid.linestyle': ':',
    'grid.linewidth': 0.5,
    'grid.alpha':     0.6,
})

In [ ]:
#| eval: false
rec = load_ecog()
clean, bad = preprocess(rec.ecog, rec.fs)
print(f'before:  mean={rec.ecog.mean():>10.1f}  std={rec.ecog.std():>8.1f}')
print(f'after:   mean={clean.mean():>10.4f}  std={clean.std():>8.1f}')
print(f'bad channels: {bad.tolist()}')

In [ ]:
#| eval: false
freqs_a, psd_a = welch(rec.ecog, fs=rec.fs, nperseg=2 * rec.fs)
freqs_b, psd_b = welch(clean,    fs=rec.fs, nperseg=2 * rec.fs)

fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
axs[0].semilogy(freqs_a, psd_a.T, lw=0.4, alpha=0.5); axs[0].set_title('before')
axs[1].semilogy(freqs_b, psd_b.T, lw=0.4, alpha=0.5); axs[1].set_title('after')
for ax in axs:
    ax.set_xlim(0, 200)
    ax.set_xlabel('frequency (Hz)')
axs[0].set_ylabel('PSD (\u03bcV\u00b2/Hz)')
fig.suptitle('Welch PSD per channel — before vs after preprocessing')
plt.tight_layout(); plt.show()

In [ ]:
#| eval: false
ch, n = 30, int(3 * rec.fs)
fig, axs = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
axs[0].plot(rec.time[:n], rec.ecog[ch, :n], lw=0.5); axs[0].set_title(f'before — channel {ch}')
axs[1].plot(rec.time[:n], clean[ch, :n],   lw=0.5); axs[1].set_title(f'after  — channel {ch}')
axs[1].set_xlabel('time (s)')
plt.tight_layout(); plt.show()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()